# Notebook 2: Predicting Antibiotic Resistance in *E. coli* Using Logistic Regression

Adapted from: 
1. Moradigaravand, D. et al. Prediction of antibiotic resistance in Escherichia coli from large-scale pan-genome data. PLOS Computational Biology 14, e1006258 (2018)
2. Orcales, F. et al. Using genomic data and machine learning to predict antibiotic resistance: A tutorial paper. PLOS Computational Biology 20, e1012579 (2024).

In Notebook 1, we used synthetic data to explore what happens when a model is 
too simple, too complex, or just right. In this notebook, we apply those same 
ideas to a real biological problem: predicting whether a strain of *Escherichia 
coli* is resistant or susceptible to an antibiotic, using genomic data as input.

This is a classification problem. Rather than predicting a continuous value like 
growth rate, we are predicting a category: resistant (R) or susceptible (S). The 
model we will use is logistic regression, a linear model adapted for classification 
tasks. By the end of this notebook, you will have trained a logistic regression 
model on real genomic data, made predictions on held-out samples, and evaluated 
how well the model performed.

## Part 1: Background

### Antibiotic resistance and why it matters

Antibiotic resistance is one of the most pressing public health challenges of our 
time. Bacteria evolve resistance to the drugs used to treat infections, and strains 
that are resistant to multiple antibiotics simultaneously are becoming increasingly 
common. Determining whether a bacterial strain is resistant to a given antibiotic 
currently requires growing the bacteria in the lab in the presence of the drug, a 
process that takes time and resources.

An alternative approach is to sequence the genome of the bacterial strain and use 
those sequences to predict resistance computationally. This is where machine 
learning comes in.

### The dataset

The data we will use in this notebook comes from a study by Moradigaravand et al. 
(2018), published in PLOS Computational Biology. The study sequenced the genomes 
of 1,936 *E. coli* isolates collected from patients and tested each isolate for 
resistance to 12 different antibiotics in the laboratory. The authors then asked 
whether machine learning models could predict those resistance outcomes from the 
genomic data alone.

One of the key findings of the study was that logistic regression, the simplest 
model they tested, performed comparably to more complex models including random 
forests and deep learning for many of the antibiotics. This is a good example of 
the principle we discussed in the Week 2 material: start with the simplest model 
that can plausibly address your question before reaching for something more complex.

We will work through a simplified version of their logistic regression analysis 
for a single antibiotic: ceftazidime (CTZ), a beta-lactam antibiotic commonly 
used to treat *E. coli* infections.

In [ ]:
# Run this cell to load all the libraries we need for this notebook.

import pandas as pd
import numpy as np
from functools import reduce

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning
warnings.filterwarnings(action='ignore', category=ConvergenceWarning)
warnings.filterwarnings(action='ignore', category=UndefinedMetricWarning)

## Part 2: Loading and exploring the metadata

We will load two datasets directly from the study's public GitHub repository. 
The first is the **metadata**, which contains information about each of the 
1,936 *E. coli* isolates: when it was collected, which sequence type it belongs 
to, and whether it was resistant or susceptible to each of the 12 antibiotics 
tested.

In [ ]:
# Load the metadata CSV directly from the study's public GitHub repository

metadata_url = ('https://raw.githubusercontent.com/Lucy-Moctezuma/ML-Tutorial-for-'
                'Antibiotic-Resistance-Predictions-for-E.-Coli/main/Datasets/Metadata.csv')

metadata = pd.read_csv(metadata_url)
metadata.head()

### Understanding the columns

The metadata contains the following columns:

- **Lane.Accession:** A unique identifier for each DNA sequence record in a 
  public database.
- **MLST:** Multi Locus Sequence Type. A classification that groups isolates 
  by genetic similarity based on seven housekeeping genes (explained below).
- **Isolate:** A unique number identifying each individual bacterial strain.
- **Year:** The year the isolate was collected from a patient.
- **Antibiotic columns (12 total):** Each column contains the result of a 
  laboratory susceptibility test for one antibiotic. Results are recorded as 
  **R** (resistant) or **S** (susceptible). Isolates classified as intermediate 
  in the original study have been grouped with resistant isolates, following 
  the approach used in the paper.

The 12 antibiotics in the dataset are listed below:

| Abbreviation | Class | Full name |
|:------------|:------|:----------|
| CTZ | Beta-lactams: Cephalosporins | Ceftazidime |
| CTX | Beta-lactams: Cephalosporins | Cefotaxime |
| CXM | Beta-lactams: Cephalosporins | Cefuroxime |
| CET | Beta-lactams: Cephalosporins | Cephalothin |
| AMP | Beta-lactams: Penicillin | Ampicillin |
| AMX | Beta-lactams: Penicillin | Amoxicillin |
| AMC | Beta-lactams: Penicillin | Amoxicillin + Clavulanate potassium |
| TZP | Beta-lactams: Piperacillin | Tazobactam |
| GEN | Aminoglycosides | Gentamicin |
| TBM | Aminoglycosides | Tobramycin |
| TMP | Antifolate | Trimethoprim |
| CIP | Fluoroquinolones | Ciprofloxacin |

In [ ]:
# How many isolates are in the dataset, and how many columns?
print("Dataset shape (rows, columns):", metadata.shape)

### Multi Locus Sequence Typing (MLST)

When we use genomic data to train a machine learning model, there is a risk 
that the model learns to recognize groups of closely related isolates rather 
than the underlying biological signal driving resistance. Isolates from the 
same patient, hospital, or outbreak may share very similar genomes simply 
because they are related, not because those genomic features cause resistance. 
If related isolates end up in both the training and test sets, the model may 
appear to perform well without having learned anything genuinely useful.

Multi Locus Sequence Typing (MLST) gives us a way to track this. MLST 
classifies bacterial isolates into sequence types (STs) based on the 
combinations of alleles they carry at seven housekeeping genes: *adk*, *fumC*, 
*gyrB*, *icd*, *mdh*, *purA*, and *recA*. Isolates with identical allele 
combinations at all seven genes belong to the same sequence type and are 
considered closely related. The figure below shows three example isolates 
and how their allele combinations determine their sequence type.

In [ ]:
# Display the MLST schematic figure

from IPython.display import Image
Image(url='https://raw.githubusercontent.com/raqueldias/ML4Bio/main/mlst_graph.jpg')


In [ ]:
# How many distinct sequence types are in the dataset?

from pandas.api.types import is_numeric_dtype

# Add "ST" prefix to make sequence types categorical rather than numeric
if is_numeric_dtype(metadata['MLST']):
    metadata['MLST'] = 'ST' + metadata['MLST'].astype(str)
    metadata['MLST'] = metadata['MLST'].map(lambda x: x.rstrip('.0'))

mlst_list = metadata['MLST'].unique()
print(f"Number of distinct sequence types: {len(mlst_list)}")

In [ ]:
# Plot the 15 most abundant sequence types in the dataset

bplotdata = (metadata['MLST']
             .value_counts()
             .sort_values(ascending=False)
             .head(15))

plt.figure(figsize=(10, 6))
bplotdata.plot(kind='barh', color='steelblue', edgecolor='gray')
plt.title("15 most common sequence types in the dataset", fontsize=14)
plt.xlabel("Number of isolates", fontsize=13)
plt.ylabel("Sequence type", fontsize=13)
plt.tight_layout()
plt.show()

**Try it** Look at the barchart. One sequence type is far more abundant than 
all the others. Which sequence type is it, and roughly how many isolates does 
it contain? Why might having many isolates from a single sequence type be a 
concern when splitting data into training and test sets?

## Part 3: Loading and exploring the gene presence/absence data

The second dataset contains information about which genes are present or absent 
in each isolate. Not all *E. coli* strains carry the same set of genes. Beyond 
the core genome, which is shared by nearly all *E. coli* isolates, each strain 
may carry a variable set of additional genes known as the **accessory genome**. 
Together, the core genome and the accessory genome make up the **pan-genome**: 
the full set of genes found across all isolates of a species.

Accessory genes are particularly relevant for antibiotic resistance because many 
resistance mechanisms are encoded by genes that are not universally present. A 
strain that carries a gene encoding a drug-inactivating enzyme, for example, may 
be resistant to a particular antibiotic while a strain lacking that gene remains 
susceptible. This makes gene presence/absence data a natural starting point for 
predicting resistance from genomic sequences.

In this dataset, each row represents one isolate and each column represents one 
gene. A value of **1** means the gene is present in that isolate and a value of 
**0** means it is absent.

In [ ]:
# Load the gene presence/absence data from the same GitHub repository

gene_url = ('https://raw.githubusercontent.com/Lucy-Moctezuma/ML-Tutorial-for-'
            'Antibiotic-Resistance-Predictions-for-E.-Coli/main/Datasets/AccessoryGene.csv')

gene_data = pd.read_csv(gene_url)
gene_data.rename(columns={'Unnamed: 0': 'Isolate'}, inplace=True)

print("Dataset shape (rows, columns):", gene_data.shape)
gene_data.head()

The gene presence/absence dataset contains 2,033 isolates and 17,198 gene 
columns. Notice that this is more isolates than in the metadata (1,936). Some 
isolates have genomic data but no accompanying resistance phenotype data, and 
we can only work with isolates that have both. We will handle this when we 
merge the two datasets in the next step.

The genes in this dataset fall into two categories:

- **Named genes:** Genes with a recognized name extracted from annotated 
  genome sequences. These are genes whose function is at least partially 
  characterized.
- **Unnamed genes (ortholog groups):** Genes that have been identified as 
  likely functional sequences but have not been assigned a name. These are 
  grouped by sequence similarity and given labels starting with "group_". 
  Their functions may not be well characterized, but their presence or absence 
  can still carry predictive signal.

In [ ]:
# Count named genes and unnamed ortholog groups

named_genes = [col for col in gene_data.columns
               if col != 'Isolate' and 'group' not in col]
unnamed_genes = [col for col in gene_data.columns
                 if 'group' in col]

print(f"Named genes:            {len(named_genes)}")
print(f"Unnamed ortholog groups: {len(unnamed_genes)}")
print(f"Total gene features:    {len(named_genes) + len(unnamed_genes)}")

**Try it** Run the cell above and look at the counts. Named genes make up 
only a small fraction of the total features in this dataset. What does this 
tell you about how well characterized the *E. coli* pan-genome is? Given what 
you know about overfitting from Notebook 1, does the ratio of features to 
isolates in this dataset raise any concerns?

### Class imbalance in the resistance labels

Before we move on to merging and modeling, it is worth noting something 
important about the resistance labels in this dataset. For most antibiotics, 
there are substantially more susceptible isolates than resistant ones. This 
is good news from a public health perspective, but it creates a practical 
challenge for machine learning.

When one class is much more common than the other, a model can achieve 
misleadingly high accuracy simply by predicting the majority class for every 
sample. For example, if 90% of isolates are susceptible to a given antibiotic, 
a model that always predicts "susceptible" would be 90% accurate without having 
learned anything useful. This is why accuracy alone is not always a reliable 
metric for imbalanced datasets, and why we will also look at recall when we 
evaluate our model in Part 8.

The cell below shows the class counts for our chosen antibiotic, CTZ.

In [ ]:
# Check the balance of resistant vs susceptible isolates for CTZ

print("Class counts for CTZ (ceftazidime):")
print(metadata['CTZ'].value_counts())
print(f"\nPercentage resistant: "
      f"{metadata['CTZ'].value_counts()['R'] / metadata['CTZ'].value_counts().sum() * 100:.1f}%")

**Try it:** Change `'CTZ'` to another antibiotic abbreviation from the table 
in Part 2, for example `'AMP'` or `'CIP'`, and re-run the cell. Which 
antibiotic has the most balanced ratio of resistant to susceptible isolates? 
Which has the most imbalanced? How might a highly imbalanced dataset affect 
your confidence in a model that reports high accuracy?

## Part 4: Merging the datasets

Now that we have loaded both datasets, we need to combine them into a single 
dataframe so that each isolate has both its resistance labels and its gene 
presence/absence features in one place. We match isolates between the two 
datasets using the isolate number, and we only keep isolates that appear in 
both datasets.

In [ ]:
# Merge the metadata and gene presence/absence data on the isolate number.
# Only isolates present in both datasets are kept (inner join).

df_list = [metadata, gene_data]
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on=['Isolate'], how='inner'),
    df_list)

# Remove the Lane.accession column, which is not needed for modeling
merged_df.drop(columns=['Lane.accession'], inplace=True)

print("Shape of merged dataset (rows, columns):", merged_df.shape)
print(f"\nIsolates in metadata:          {metadata.shape[0]}")
print(f"Isolates in gene data:         {gene_data.shape[0]}")
print(f"Isolates in merged dataset:    {merged_df.shape[0]}")

After merging, we have 1,936 isolates, matching the metadata exactly. The 
isolates present in the gene data but not in the metadata have been dropped, 
since we cannot train or evaluate a model without knowing the resistance 
outcome for each isolate.

The merged dataset now contains:
- 1 MLST column
- 1 isolate identifier column
- 12 resistance label columns (one per antibiotic)
- 1 year of isolation column
- 17,198 gene presence/absence columns

In the next step, we will focus on a single antibiotic (CTZ) and prepare the 
data for modeling.

## Part 5: Preparing the data for modeling

Before we can train a logistic regression model, we need to do three things:

1. Select the features and label we will use
2. Remove any isolates with missing data for our chosen antibiotic
3. Split the data into a training set and a test set

### Selecting features and labels

We will use two types of features to predict CTZ resistance:

- **Year of isolation (Y):** The year each isolate was collected from a patient. 
  Resistance patterns in bacterial populations change over time as resistant 
  strains spread, so the year of isolation can carry predictive signal.
- **Gene presence/absence (G):** Whether each of the 17,198 accessory genes is 
  present or absent in each isolate. This is the primary source of biological 
  signal for predicting resistance, since many resistance mechanisms are encoded 
  by specific accessory genes.

This combination of features is referred to as **GY** in the Moradigaravand 
et al. paper, and it was used in the best-performing models for most antibiotics.

In [ ]:
# Select the features and label for CTZ

drug = 'CTZ'

# Keep the MLST column, the drug label, the year column, and all gene columns
df_list = [merged_df[['MLST', 'Isolate', drug, 'Year']],
           merged_df.iloc[:, 15:]]
drug_df = pd.concat(df_list, axis=1)

# Drop rows with missing resistance labels for this drug
drug_df = drug_df.dropna()

# Separate features and labels
labels   = drug_df[drug]
features = drug_df.drop(columns=[drug])

print(f"Isolates available for {drug}: {drug_df.shape[0]}")
print(f"Number of features: {features.shape[1] - 2}")  # excluding MLST and Isolate
print(f"\nClass counts:")
print(labels.value_counts())

### Splitting into training and test sets

We split the data into a training set, which the model will learn from, and 
a test set, which will be held out and used only to evaluate the final model. 
We use 80% of the data for training and 20% for testing.

Notice the `stratify=labels` argument in the cell below. This ensures that 
the proportion of resistant and susceptible isolates is the same in both the 
training and test sets. Without stratification, random splitting could by 
chance put most of the resistant isolates into one set, which would make 
training or evaluation unreliable given how few resistant isolates there are.

In [ ]:
# Split into training and test sets
# 80% training, 20% test, stratified by resistance label

features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print(f"Training samples: {len(features_train)}")
print(f"Test samples:     {len(features_test)}")
print(f"\nClass counts in training set:")
print(labels_train.value_counts())
print(f"\nClass counts in test set:")
print(labels_test.value_counts())

**Try it:** The test size is currently set to `0.20`, meaning 20% of isolates 
are held out for testing. Change `test_size=0.20` to `test_size=0.50` and 
re-run the cell. How does this affect the number of training samples? Based 
on what you learned in Notebook 1 about the relationship between sample size 
and model performance, why might reducing the training set size be a concern, 
especially for a dataset with this many features?

Before we train the model, we need to create the final feature dataframes 
for training and testing. We drop the MLST and Isolate columns here since 
they are identifiers rather than biological features, and we keep only the 
year and gene presence/absence columns that the model will actually use.

In [ ]:
# Create final feature dataframes for training and testing
# Drop MLST and Isolate columns, these are identifiers, not features

features_train_final = features_train.drop(columns=['MLST', 'Isolate'])
features_test_final  = features_test.drop(columns=['MLST', 'Isolate'])

print("Training feature matrix shape:", features_train_final.shape)
print("Test feature matrix shape:    ", features_test_final.shape)

## Part 6: Fitting the logistic regression model

We are now ready to train our logistic regression model. Recall from the Week 2 
material that logistic regression is a linear model adapted for classification. 
Rather than predicting a continuous value, it predicts the probability that a 
sample belongs to one of two classes. In our case, those classes are resistant 
(R) and susceptible (S).

The equation at the heart of logistic regression converts a weighted sum of the 
input features into a probability between 0 and 1:

$$P = \frac{e^{\hat{\beta}_{0}+\hat{\beta}_{j}X}}{1+e^{\hat{\beta}_{0}+\hat{\beta}_{j}X}}$$

Where:
- **P** is the predicted probability of resistance
- **$\hat{\beta}_0$** is the intercept
- **$\hat{\beta}_j$** are the coefficients, one for each feature
- **X** represents our feature values (year and gene presence/absence)

If P is greater than 0.5 the model predicts resistant (R), and if P is less 
than 0.5 it predicts susceptible (S). You do not need to memorize this equation. 
What matters is understanding that the model is learning one coefficient per 
feature, and that those coefficients represent how strongly each feature pushes 
the prediction toward resistance or susceptibility.

Note that we have 17,199 features in total, which means the model is estimating 
17,199 coefficients. This is a good illustration of why logistic regression is 
considered a linear model even when applied to very high-dimensional data: the 
prediction is still a weighted sum of the inputs, just with many more weights 
to estimate.

In [ ]:
# Train the logistic regression model on the training data.
# class_weight='balanced' adjusts for the imbalance between resistant and 
# susceptible isolates by giving more weight to the minority class (R) 
# during training.

model = LogisticRegression(
    random_state=42,
    max_iter=500,
    class_weight='balanced'
)

model.fit(features_train_final, labels_train)
print("Model training complete.")
print(f"Classes predicted by the model: {model.classes_}")

### A brief look at the model coefficients

Once the model is trained, we can inspect the coefficients it has learned. 
Each coefficient corresponds to one feature and tells us the direction of 
that feature's association with resistance: a positive coefficient means 
the presence of that gene (or a later year) pushes the prediction toward 
resistance, while a negative coefficient pushes it toward susceptibility.

With 17,199 coefficients it is not practical to examine them all, but we 
can look at the genes most strongly associated with resistance and 
susceptibility in either direction.

In [ ]:
# Extract coefficients and match them to feature names

coef_df = pd.DataFrame({
    'feature':     features_train_final.columns,
    'coefficient': model.coef_[0]
})

# Sort by coefficient value
coef_df = coef_df.sort_values('coefficient', ascending=False)

print("Top 10 features most associated with RESISTANCE (positive coefficients):")
print(coef_df.head(10).to_string(index=False))
print()
print("Top 10 features most associated with SUSCEPTIBILITY (negative coefficients):")
print(coef_df.tail(10).to_string(index=False))

The genes with the largest positive coefficients are those whose presence is 
most strongly associated with resistance to CTZ in this dataset. Many of these 
are likely to be known beta-lactamase genes or other resistance-associated 
accessory genes. The genes with the largest negative coefficients are those 
whose presence is most strongly associated with susceptibility.

Keep in mind that these associations are not necessarily causal. A gene may 
have a large coefficient because it is genuinely involved in resistance, or 
because it tends to co-occur with other resistance-associated genes in the 
same isolates. Interpreting model coefficients requires biological knowledge 
alongside the statistical output.

**Try it:** Look at the top 10 features most associated with resistance. 
Are any of them named genes (i.e., do they have a recognizable gene name 
rather than a "group_" label)? Based on what you know about beta-lactam 
resistance mechanisms, do any of the named genes make biological sense as 
resistance-associated features?

## Part 7: Making predictions

Now that the model has been trained and its coefficients estimated, we can 
use it to make predictions on the test set. Remember that the test set 
contains isolates the model has never seen during training. This is the 
critical step: we want to know not just how well the model fits the 
training data, but how well it generalizes to new isolates.

The model first computes a predicted probability of resistance for each 
test isolate, then assigns a label of R or S based on whether that 
probability is above or below 0.5.

In [ ]:
# Generate predicted labels for the test set

labels_pred = model.predict(features_test_final)

# Show the distribution of predicted labels
unique, counts = np.unique(labels_pred, return_counts=True)
print("Predicted label counts:")
for label, count in zip(unique, counts):
    print(f"  {label}: {count}")

In [ ]:
# Compare 20 randomly sampled predicted labels to the actual labels
# We use a fixed random seed so the output is the same every time you run this cell

np.random.seed(7)
sample_idx = np.random.choice(len(labels_pred), size=25, replace=False)

print("Predicted labels (25 randomly sampled test isolates):")
print(labels_pred[sample_idx])
print("\nActual labels (25 randomly sampled test isolates):")
print(np.array(labels_test)[sample_idx])

Scanning through these 25 predictions, you can see that most match the 
actual labels, but some do not. A few resistant isolates have been predicted 
as susceptible, and possibly a few susceptible isolates have been predicted 
as resistant. To get a clearer picture of where the model is succeeding and 
where it is failing, we need a more systematic evaluation, which is what 
Part 8 covers.

**Try it:** Count how many of the 25 predictions match the actual 
label and how many do not. Based on this small sample, does the model seem 
to make more errors on resistant isolates or susceptible isolates? Why might 
that pattern occur given what you learned about class imbalance in Part 3?

## Part 8: Evaluating the model

Looking at individual predictions gives us a rough sense of how the model 
is doing, but we need a more systematic way to evaluate performance across 
all 376 test isolates. We will use three tools: a confusion matrix, accuracy, 
and recall.

### The confusion matrix

A confusion matrix summarizes all of the model's predictions in a single 
table. For a two-class problem like ours, it has four cells:

- **True Positives (TP):** Resistant isolates correctly predicted as resistant
- **True Negatives (TN):** Susceptible isolates correctly predicted as susceptible
- **False Positives (FP):** Susceptible isolates incorrectly predicted as resistant
- **False Negatives (FN):** Resistant isolates incorrectly predicted as susceptible

In a clinical context, false negatives are particularly costly: a resistant 
isolate predicted as susceptible means a patient might be treated with a drug 
that will not work.

In [ ]:
# Plot the confusion matrix

cm = confusion_matrix(labels_test, labels_pred, labels=['R', 'S'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['R', 'S'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title("Confusion matrix: CTZ resistance prediction", fontsize=13)
plt.tight_layout()
plt.show()

### Accuracy, recall, and what they each tell us

From the confusion matrix we can calculate several performance metrics. We 
will focus on two: accuracy and recall.

**Accuracy** is the proportion of all predictions that were correct:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Accuracy is intuitive but can be misleading when classes are imbalanced. 
A model that predicted "susceptible" for every single isolate would achieve 
high accuracy simply because most isolates are susceptible, even though it 
would completely fail to detect any resistant cases.

**Recall** measures how well the model identifies each class. It is calculated 
separately for resistant and susceptible isolates:

$$\text{Recall for R} = \frac{TP}{TP + FN}$$

$$\text{Recall for S} = \frac{TN}{TN + FP}$$

Recall for R tells us what fraction of the truly resistant isolates the model 
correctly identified. In a clinical setting where missing a resistant isolate 
has serious consequences, this is often the most important metric to examine.

In [ ]:
# Calculate and display accuracy and recall

report = classification_report(labels_test, labels_pred, output_dict=True)

accuracy  = report['accuracy']
r_recall  = report['R']['recall']
s_recall  = report['S']['recall']
r_precision = report['R']['precision']
s_precision = report['S']['precision']

print(f"Accuracy:                {accuracy:.3f}")
print(f"Recall for R:            {r_recall:.3f}")
print(f"Recall for S:            {s_recall:.3f}")
print(f"Precision for R:         {r_precision:.3f}")
print(f"Precision for S:         {s_precision:.3f}")

Look carefully at these three numbers together. Accuracy alone gives an 
optimistic picture of model performance. But recall for R tells a more 
nuanced story: the model does not correctly identify all resistant isolates, 
even though its overall accuracy is high.

This pattern is a direct consequence of class imbalance. With many more 
susceptible isolates than resistant ones in the training data, the model has 
seen far more examples of susceptibility and has learned that predicting 
susceptible is a relatively safe bet. Even with `class_weight='balanced'` 
helping to correct for this, some of that imbalance still shows up in the 
results.

This is exactly the concern raised in the Moradigaravand et al. paper, and 
it is one of the reasons they emphasize recall for resistance as a key metric 
alongside accuracy.

**Try it:** Look at the confusion matrix. How many resistant isolates were 
incorrectly predicted as susceptible (false negatives)? How many susceptible 
isolates were incorrectly predicted as resistant (false positives)? Which 
type of error do you think is more consequential in a clinical setting where 
the goal is to identify resistant infections, and why?

### Connecting back to the Moradigaravand et al. findings

Recall from Part 1 that one of the key findings of the Moradigaravand et al. 
paper was that logistic regression performed comparably to more complex models 
including random forests and deep learning for many of the antibiotics tested. 
The model you just trained is a simplified version of their logistic regression 
analysis, using the same dataset and the same GY feature combination.

This is a concrete example of the complexity escalation principle from the 
Week 2 material: a simple linear model can be a strong and appropriate choice 
even for a high-dimensional biological dataset, and reaching for a more complex 
model is only justified when the simpler one demonstrably falls short.

**Try it:** Change `drug = 'CTZ'` in Part 5 to a different antibiotic, for 
example `'CIP'` or `'GEN'`, and re-run all cells from Part 5 onward. How do 
the accuracy and recall values change? Does the model perform better or worse 
for antibiotics with more balanced class distributions?